In this project, you will be working with J1939 fault code data and vehicle onboard diagnostic data to try and predict an upcoming full derate.

J1939 is a communications protocol used in heavy-duty vehicles (like trucks, buses, and construction equipment) to allow different electronic control units (ECUs), like the engine, transmission, and brake systems, to talk to each other. Fault codes in this system follow a standard format so that mechanics and diagnostic tools can understand what's wrong, no matter the make or model.

These fault codes have two parts. First, an SPN (Suspect Parameter Number), which identifies what system or component is having the issue. Second, an FMI (Failure Mode Identifier), which explains how the system is failing (too high, too low, short circuit, etc.).

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from imblearn.over_sampling import SMOTE

In [2]:
faults = pd.read_csv("../data/J1939Faults.csv", low_memory=False)
diagnostics = pd.read_csv("../data/VehicleDiagnosticOnboardData.csv")

In [3]:
faults.head()

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
0,1,990349,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,NaN,unknown,unknown,unknown,unknown,0,111,17,True,2,NaN,1439,105354361,38.857638,-84.626851,2015-02-21 11:34:25.000
1,2,990360,2015-02-21 11:34:34.000,NaN,NaN,unknown,unknown,unknown,unknown,11,629,12,True,127,NaN,1439,105354361,38.857638,-84.626851,2015-02-21 11:35:10.000
2,3,990364,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,NaN,unknown,unknown,unknown,unknown,11,1807,2,False,127,NaN,1369,105336226,41.421250,-87.767361,2015-02-21 11:35:26.000
3,4,990370,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,NaN,unknown,unknown,unknown,unknown,11,1807,2,True,127,NaN,1369,105336226,41.421018,-87.767361,2015-02-21 11:36:08.000
4,5,990416,2015-02-21 11:39:41.000,NaN,NaN,22281684P01*22357957P01*22362082P01*,13063430,0USA13_13_0415_2238A,VOLVO,0,4364,17,False,2,NaN,1674,105427130,38.416481,-89.442638,2015-02-21 11:39:37.000


In [4]:
faults.shape

(1187335, 20)

In [5]:
diagnostics.shape

(12821626, 4)

In [6]:
faults.columns

Index(['RecordID', 'ESS_Id', 'EventTimeStamp', 'eventDescription',
       'actionDescription', 'ecuSoftwareVersion', 'ecuSerialNumber',
       'ecuModel', 'ecuMake', 'ecuSource', 'spn', 'fmi', 'active',
       'activeTransitionCount', 'faultValue', 'EquipmentID', 'MCTNumber',
       'Latitude', 'Longitude', 'LocationTimeStamp'],
      dtype='object')

In [7]:
diagnostics.columns

Index(['Id', 'Name', 'Value', 'FaultId'], dtype='object')

In [8]:
diagnostics['Name'].unique()

array(['IgnStatus', 'EngineOilPressure', 'EngineOilTemperature',
       'TurboBoostPressure', 'EngineLoad', 'AcceleratorPedal',
       'IntakeManifoldTemperature', 'FuelRate', 'FuelLtd', 'EngineRpm',
       'LampStatus', 'BarometricPressure', 'FuelLevel', 'Speed',
       'EngineTimeLtd', 'CruiseControlSetSpeed', 'CruiseControlActive',
       'EngineCoolantTemperature', 'ParkingBrake',
       'SwitchedBatteryVoltage', 'DistanceLtd', 'Throttle',
       'FuelTemperature', 'ServiceDistance'], dtype=object)

In [9]:
# Convert Diagnostics Data from Long to Wide Format
diagnostics_wide = diagnostics.pivot_table(
    index='FaultId',
    columns='Name',
    values='Value',
    aggfunc='first'
).reset_index()

In [10]:
diagnostics_wide.shape

(1187335, 25)

In [11]:
diagnostics_wide.columns

Index(['FaultId', 'AcceleratorPedal', 'BarometricPressure',
       'CruiseControlActive', 'CruiseControlSetSpeed', 'DistanceLtd',
       'EngineCoolantTemperature', 'EngineLoad', 'EngineOilPressure',
       'EngineOilTemperature', 'EngineRpm', 'EngineTimeLtd', 'FuelLevel',
       'FuelLtd', 'FuelRate', 'FuelTemperature', 'IgnStatus',
       'IntakeManifoldTemperature', 'LampStatus', 'ParkingBrake',
       'ServiceDistance', 'Speed', 'SwitchedBatteryVoltage', 'Throttle',
       'TurboBoostPressure'],
      dtype='object', name='Name')

In [12]:
diagnostics_wide.isna().sum().sort_values(ascending=False)

Name
ServiceDistance              1187120
SwitchedBatteryVoltage       1073276
FuelTemperature               888225
ParkingBrake                  787363
Throttle                      766832
FuelLevel                     684540
AcceleratorPedal              655446
CruiseControlActive           612419
CruiseControlSetSpeed         610877
EngineTimeLtd                 605969
TurboBoostPressure            603984
EngineOilTemperature          603423
Speed                         603419
FuelLtd                       602140
FuelRate                      602098
EngineLoad                    601714
DistanceLtd                   601516
BarometricPressure            601359
EngineCoolantTemperature      601264
EngineOilPressure             601091
IntakeManifoldTemperature     601044
EngineRpm                     600414
IgnStatus                     578881
FaultId                            0
LampStatus                         0
dtype: int64

In [13]:
# Separate Numeric and Categorical Variables
numeric_features = [
    'AcceleratorPedal',
    'BarometricPressure',
    'CruiseControlSetSpeed',
    'DistanceLtd',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'EngineTimeLtd',
    'FuelLevel',
    'FuelLtd',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'ServiceDistance',
    'Speed',
    'SwitchedBatteryVoltage',
    'Throttle',
    'TurboBoostPressure'
]

#categorical_variable
categorical_features = [
    'CruiseControlActive',
    'IgnStatus',
    'LampStatus',
    'ParkingBrake'
]

In [14]:
# Now build the numeric pipeline
numeric_transformer = Pipeline(
    steps=[
        ('scaler', MaxAbsScaler()),
        ('imputer', IterativeImputer(random_state=321))
    ]
)

In [15]:
# Categorical pipeline
categorical_transformer = Pipeline(
    steps=[
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [16]:
# numeric and categorical
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


In [17]:
for col in numeric_features:
    diagnostics_wide[col] = (
        diagnostics_wide[col]
        .astype(str)
        .str.replace(',', '', regex=False)
    )

    diagnostics_wide[col] = pd.to_numeric(
        diagnostics_wide[col],
        errors='coerce'
    )

In [18]:
diagnostics_prepared_array = preprocessor.fit_transform(diagnostics_wide)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [19]:
# Get Feature Names
feature_names = preprocessor.get_feature_names_out()

In [20]:
# Convert array to dataframe
diagnostics_prepared_df = pd.DataFrame(
    diagnostics_prepared_array,
    columns=feature_names
)

In [21]:
# Add FaultId back
diagnostics_prepared_df['FaultId'] = diagnostics_wide['FaultId'].values

In [22]:
diagnostics_prepared_df.head()

,num__AcceleratorPedal,num__BarometricPressure,num__CruiseControlSetSpeed,num__DistanceLtd,num__EngineCoolantTemperature,num__EngineLoad,num__EngineOilPressure,num__EngineOilTemperature,num__EngineRpm,num__EngineTimeLtd,...,cat__LampStatus_617,cat__LampStatus_62463,cat__LampStatus_63487,cat__LampStatus_65523,cat__LampStatus_65535,cat__LampStatus_9,cat__ParkingBrake_False,cat__ParkingBrake_True,cat__ParkingBrake_nan,FaultId
0,0.000000,0.000098,0.914530,0.045421,0.053234,0.106796,0.000000,0.000045,0.000000,0.003565,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
1,0.027309,0.000115,0.822238,0.038696,0.088231,0.288779,0.003623,0.000105,0.005448,0.015751,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2
2,0.027309,0.000115,0.822238,0.038696,0.088231,0.288779,0.003623,0.000105,0.005448,0.015751,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3
3,0.027309,0.000115,0.822238,0.038696,0.088231,0.288779,0.003623,0.000105,0.005448,0.015751,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,0.027309,0.000115,0.822238,0.038696,0.088231,0.288779,0.003623,0.000105,0.005448,0.015751,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5


In [23]:
# Indentify the hiding features
[col for col in diagnostics_prepared_df.columns if 'IgnStatus' in col]

['cat__IgnStatus_False', 'cat__IgnStatus_True', 'cat__IgnStatus_nan']

In [24]:
# Merge Faults and Processed Diagnostics Data
merged_data = faults.merge(
    diagnostics_prepared_df,
    left_on='RecordID',
    right_on='FaultId',
    how='inner'
)

In [25]:
merged_data.shape

(1187335, 87)

In [26]:
merged_data.head()

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,...,cat__LampStatus_617,cat__LampStatus_62463,cat__LampStatus_63487,cat__LampStatus_65523,cat__LampStatus_65535,cat__LampStatus_9,cat__ParkingBrake_False,cat__ParkingBrake_True,cat__ParkingBrake_nan,FaultId
0,1,990349,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,NaN,unknown,unknown,unknown,unknown,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
1,2,990360,2015-02-21 11:34:34.000,NaN,NaN,unknown,unknown,unknown,unknown,11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2
2,3,990364,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,NaN,unknown,unknown,unknown,unknown,11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3
3,4,990370,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,NaN,unknown,unknown,unknown,unknown,11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,990416,2015-02-21 11:39:41.000,NaN,NaN,22281684P01*22357957P01*22362082P01*,13063430,0USA13_13_0415_2238A,VOLVO,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5


In [27]:
# create the current derate flag.
merged_data['full_derate'] = (merged_data['spn'] == 5246).astype(int)


In [28]:
# Remove rows where the truck is already in full derate
model_data = merged_data[merged_data['full_derate'] == 0].copy()

In [29]:
model_data['full_derate'].value_counts()

full_derate
0    1186140
Name: count, dtype: int64

In [30]:
# Convert EventTimeStamp to datetime
model_data['EventTimeStamp'] = pd.to_datetime(
    model_data['EventTimeStamp']
)

# Sort events chronologically
model_data = model_data.sort_values(
    ['EquipmentID', 'EventTimeStamp']
)

In [31]:
# Remove service-location records
service_locations = [
    (36.0666667, -86.4347222),
    (35.5883333, -86.4438888),
    (36.1950, -83.174722)
]

threshold = 0.05

for lat, lon in service_locations:
    model_data = model_data[
        ~(
            (model_data['Latitude'].between(lat - threshold, lat + threshold)) &
            (model_data['Longitude'].between(lon - threshold, lon + threshold))
        )
    ]


In [32]:
model_data.shape

(1049698, 88)

In [33]:
# Sort Events by EquipmentID and Time
model_data = model_data.sort_values(
    ['EquipmentID', 'EventTimeStamp']
)

In [34]:
# Identify Future Full Derate Events
derate_events = merged_data[merged_data['full_derate'] == 1][
    ['EquipmentID', 'EventTimeStamp']
].rename(columns={'EventTimeStamp': 'next_derate_time'})

In [35]:
# Remove Existing Future Derate Timestamp Column
model_data = model_data.drop(
    columns=['next_derate_time'],
    errors='ignore'
)

In [36]:
# Make sure both timestamp columns are datetime
model_data['EventTimeStamp'] = pd.to_datetime(model_data['EventTimeStamp'])
derate_events['next_derate_time'] = pd.to_datetime(derate_events['next_derate_time'])

In [37]:
# Merge future derate timestamps into modeling data
model_data = pd.merge_asof(
    model_data.sort_values('EventTimeStamp'),
    derate_events.sort_values('next_derate_time'),
    by='EquipmentID',
    left_on='EventTimeStamp',
    right_on='next_derate_time',
    direction='forward'
)

In [38]:
model_data[['EquipmentID', 'EventTimeStamp', 'next_derate_time']].head(20)

,EquipmentID,EventTimeStamp,next_derate_time
0,2015,2000-03-18 19:14:10,NaT
1,2015,2000-03-18 19:14:10,NaT
2,2015,2000-03-18 19:20:47,NaT
3,2015,2000-03-18 19:20:47,NaT
4,1849,2000-03-19 02:59:58,NaT
5,1849,2000-03-19 03:58:23,NaT
6,2283,2000-03-19 07:32:53,NaT
7,2283,2000-03-19 07:34:31,NaT
8,2034,2000-03-19 08:40:03,NaT
9,2034,2000-03-19 08:40:03,NaT


In [39]:
# Check Rows with Matched Future Derate Events
model_data[
    model_data['next_derate_time'].notna()
][['EquipmentID', 'EventTimeStamp', 'next_derate_time']].head(20)

,EquipmentID,EventTimeStamp,next_derate_time
16,1961,2000-03-19 10:51:28,2019-02-14 13:46:15
72,1970,2000-03-19 18:56:09,2019-04-29 05:02:21
73,1970,2000-03-19 18:56:09,2019-04-29 05:02:21
215,1827,2010-12-31 22:04:03,2019-01-21 09:01:38
231,302,2010-12-31 23:04:15,2020-01-06 10:13:57
256,1739,2011-01-01 00:03:16,2018-02-18 18:47:52
272,1856,2011-01-01 00:03:25,2019-03-22 19:07:58
274,1917,2011-01-01 00:03:29,2017-01-17 18:05:22
275,1751,2011-01-01 00:03:34,2011-01-01 00:03:34
276,1751,2011-01-01 00:03:35,2019-02-15 06:41:35


In [40]:
# Remove rows where the current event is the derate event itself
model_data = model_data[
    model_data['EventTimeStamp'] != model_data['next_derate_time']
]

### Why We Removed Current Derate Event Matches

After merging future full derate timestamps into the modeling dataset, some rows contained identical values for `EventTimeStamp` and `next_derate_time`. These rows represent events where the truck was already in a full derate state at the current timestamp.

For predictive maintenance modeling, the goal is to predict future derates before they occur, rather than identify events where the derate has already happened. Keeping these rows could introduce target leakage and allow the model to learn the failure event itself instead of the warning patterns leading up to failure.

Therefore, rows where the current event timestamp matched the derate timestamp were removed to ensure the model focuses only on pre-derate behavior and early warning signals.

In [41]:
# Create Fault Count Feature per Truck

model_data['fault_count_per_truck'] = (
    model_data.groupby('EquipmentID').cumcount()
)

In [42]:
# Create Count of Previous SPN Occurrences per Truck
model_data['spn_occurrence_count'] = (
    model_data.groupby(['EquipmentID', 'spn']).cumcount()
)

In [43]:
# Review New Engineered Features

model_data[
    ['EquipmentID', 'EventTimeStamp', 'spn', 
     'fault_count_per_truck', 'spn_occurrence_count']
].head(20)

,EquipmentID,EventTimeStamp,spn,fault_count_per_truck,spn_occurrence_count
0,2015,2000-03-18 19:14:10,96,0,0
1,2015,2000-03-18 19:14:10,829,1,0
2,2015,2000-03-18 19:20:47,829,2,1
3,2015,2000-03-18 19:20:47,96,3,1
4,1849,2000-03-19 02:59:58,792,0,0
5,1849,2000-03-19 03:58:23,792,1,1
6,2283,2000-03-19 07:32:53,37,0,0
7,2283,2000-03-19 07:34:31,37,1,1
8,2034,2000-03-19 08:40:03,96,0,0
9,2034,2000-03-19 08:40:03,829,1,0


In [44]:
# Check Rows with Matched Future Derate Events
model_data[
    model_data['next_derate_time'].notna()
][['EquipmentID', 'EventTimeStamp', 'next_derate_time']].head(10)

,EquipmentID,EventTimeStamp,next_derate_time
16,1961,2000-03-19 10:51:28,2019-02-14 13:46:15
72,1970,2000-03-19 18:56:09,2019-04-29 05:02:21
73,1970,2000-03-19 18:56:09,2019-04-29 05:02:21
215,1827,2010-12-31 22:04:03,2019-01-21 09:01:38
231,302,2010-12-31 23:04:15,2020-01-06 10:13:57
256,1739,2011-01-01 00:03:16,2018-02-18 18:47:52
272,1856,2011-01-01 00:03:25,2019-03-22 19:07:58
274,1917,2011-01-01 00:03:29,2017-01-17 18:05:22
276,1751,2011-01-01 00:03:35,2019-02-15 06:41:35
280,2007,2011-01-01 00:04:07,2018-04-07 05:15:00


In [45]:
# Calculate hours until the next future derate event
model_data['hours_to_derate'] = (
    model_data['next_derate_time'] - model_data['EventTimeStamp']
).dt.total_seconds() / 3600

In [46]:
# Filter Modeling Data to Near-Term Future Derate Events

model_data_filtered = model_data[
    (model_data['hours_to_derate'].isna()) |
    (model_data['hours_to_derate'] <= 24)
].copy()

## Filter Modeling Data to Near-Term Future Derate Events

Exploratory analysis of the `hours_to_derate` variable revealed that many observations were associated with future derate events occurring months or even years after the current fault event. Although these observations were labeled as non-derate cases within the short prediction windows, they may still contain long-term degradation behavior and recurring fault activity.

This likely introduced substantial overlap between derate and non-derate operational patterns, contributing to high false-positive rates across multiple models.

To reduce this overlap and improve label relevance for near-term prediction, the modeling dataset was filtered to retain only observations associated with future derate events occurring within 24 hours of the current event, while preserving rows without future derate information. This filtering step creates a more prediction-focused dataset intended to better distinguish immediate pre-derate operational behavior from normal operating conditions.

## Engineering High-Risk SPN and Lamp Status Features

Additional exploratory analysis of SPN and lamp-status behavior revealed that several highly important operational fault patterns appeared frequently in both derate and non-derate states. This substantial overlap likely contributed to the high false-positive rates observed across multiple classification models.

To further investigate whether certain SPNs and warning lamp states were more strongly associated with near-term derate events, SPN-specific and lamp-status-specific derate rates were examined using the filtered near-term dataset. The analysis identified subsets of SPNs and lamp-status conditions with substantially higher derate probabilities compared with the overall operational population.

Based on these observations, targeted high-risk SPN and high-risk lamp-status indicator features were engineered to help the model better distinguish higher-risk operational fault patterns from common low-risk conditions that dominate both derate and non-derate classes.

In addition, an interaction feature combining the high-risk SPN and high-risk lamp-status indicators was created to investigate whether simultaneous occurrence of high-risk fault and warning-state behavior improves separability of near-term derate events under chronological validation.

In [47]:
# Create List of High-Risk SPNs Based on Derate Rate Analysis
high_risk_spn = [
    228, 74, 5745, 252, 1787,
    2017, 6802, 4376, 5394
]

# Create High-Risk SPN Indicator Feature
model_data_filtered['high_risk_spn_flag'] = (
    model_data_filtered['spn']
    .isin(high_risk_spn)
    .astype(int)
)

In [48]:
# Verify High-Risk SPN Feature Was Added Successfully

'high_risk_spn_flag' in model_data_filtered.columns

True

In [49]:
# Create High-Risk Lamp Status Indicator Feature

model_data_filtered['high_risk_lamp_flag'] = (
    model_data_filtered['cat__LampStatus_18431']
).astype(int)

In [50]:
# Verify High-Risk Lamp Status Indicator Feature Was Added Successfully

'high_risk_lamp_flag' in model_data_filtered.columns

True

In [51]:
# Create Combined High-Risk SPN and Lamp Interaction Feature

model_data_filtered['risk_spn_lamp_interaction'] = (
    model_data_filtered['high_risk_spn_flag'] *
    model_data_filtered['high_risk_lamp_flag']
)

In [52]:
# Create 2–6 hour future derate target
model_data_filtered['derate_2_6_hr'] = (
    (model_data_filtered['hours_to_derate'] >= 2) &
    (model_data_filtered['hours_to_derate'] < 6)
).astype(int)

In [53]:
model_data_filtered['derate_6_12_hr'] = (
    (model_data_filtered['hours_to_derate'] >= 6) &
    (model_data_filtered['hours_to_derate'] < 12)
).astype(int)

In [54]:
model_data_filtered['derate_12_24_hr'] = (
    (model_data_filtered['hours_to_derate'] >= 12) &
    (model_data_filtered['hours_to_derate'] < 24)
).astype(int)

In [55]:
# Check the number of positive cases in each future derate window
model_data_filtered[
    ['derate_2_6_hr',
     'derate_6_12_hr',
     'derate_12_24_hr']
].sum()

derate_2_6_hr       846
derate_6_12_hr      718
derate_12_24_hr    1074
dtype: int64

In [56]:
# Create time-based features from event timestamps
model_data_filtered['year'] = model_data_filtered['EventTimeStamp'].dt.year
model_data_filtered['month'] = model_data_filtered['EventTimeStamp'].dt.month
model_data_filtered['weekday'] = model_data_filtered['EventTimeStamp'].dt.weekday
model_data_filtered['hour'] = model_data_filtered['EventTimeStamp'].dt.hour

In [57]:
# Calculate time since the previous fault event for each truck
model_data_filtered['min_since_last_fault'] = (
    model_data_filtered.groupby('EquipmentID')['EventTimeStamp']
    .diff()
    .dt.total_seconds() / 60
)

In [58]:
model_data_filtered[
    ['EquipmentID', 'EventTimeStamp', 'min_since_last_fault']
].head(10)

,EquipmentID,EventTimeStamp,min_since_last_fault
0,2015,2000-03-18 19:14:10,NaN
1,2015,2000-03-18 19:14:10,0.000000
2,2015,2000-03-18 19:20:47,6.616667
3,2015,2000-03-18 19:20:47,0.000000
4,1849,2000-03-19 02:59:58,NaN
5,1849,2000-03-19 03:58:23,58.416667
6,2283,2000-03-19 07:32:53,NaN
7,2283,2000-03-19 07:34:31,1.633333
8,2034,2000-03-19 08:40:03,NaN
9,2034,2000-03-19 08:40:03,0.000000


# Random Forest Model — Predicting Full Derate Within 2–6 Hours

In [59]:
predictive_variable = model_data_filtered.drop(
    columns=[
        'derate_2_6_hr',
        'derate_6_12_hr',
        'derate_12_24_hr',
        'hours_to_derate',
        'next_derate_time',
        'next_derate_time_x',
        'next_derate_time_y',
        'EventTimeStamp',
        'LocationTimeStamp',
        'RecordID',
        'FaultId',
        'eventDescription',
        'ecuSoftwareVersion',
        'ecuSerialNumber',
        'ecuModel',
        'ecuMake',
        'EquipmentID',
        'actionDescription',
        'faultValue'
    ],
    errors='ignore'
)


In [60]:
# Define the prediction target and predictive variables
y = model_data_filtered['derate_2_6_hr']

# Use features from predictive_variable
X = predictive_variable

# Check shapes
y.shape, X.shape

((792176,), (792176, 86))

In [61]:
# Check for any remaining object columns
predictive_variable.select_dtypes(include='object').columns

Index([], dtype='object')

In [62]:
# Sort Data Chronologically by Event Timestamp
model_data_filtered = model_data_filtered.sort_values('EventTimeStamp')

In [63]:

# Verify Predictor and Target Shapes for 2–6 Hour Model
X.shape, y.shape

((792176, 86), (792176,))

## Comparison of Modeling Dataset Before and After Filtering

The original modeling dataset contained 1,049,350 observations with 83 predictor features. However, exploratory analysis of the `hours_to_derate` variable showed that many rows were associated with future derate events occurring months or even years after the current fault event. These long-term future derate observations may introduce substantial overlap between derate and non-derate classes.

To reduce this overlap and improve label relevance for near-term prediction, the dataset was filtered to retain only rows associated with future derate events occurring within 24 hours of the current event, while preserving rows without future derate information.

Additional exploratory analysis further revealed that several highly important SPNs and lamp-status conditions appeared frequently in both derate and non-derate operational states. To help the model better distinguish higher-risk operational fault behavior, targeted high-risk SPN and high-risk lamp-status indicator features were engineered based on elevated derate-rate patterns identified within the filtered dataset. An interaction feature combining the high-risk SPN and lamp-status indicators was also created to investigate whether simultaneous high-risk fault and warning-state behavior improves near-term derate separability.

After filtering and feature engineering, the modeling dataset contained 792,176 observations with 85 predictor features. These refinements focus the analysis on more prediction-relevant operational behavior and may improve the model’s ability to distinguish near-term derate events from normal operating conditions under chronological validation.

In [64]:
# Check Class Balance for the 2–6 Hour Derate Target
y.value_counts()

derate_2_6_hr
0    791330
1       846
Name: count, dtype: int64

In [65]:
# Check Proportion of Positive and Negative Derate Cases
y.value_counts(normalize=True)

derate_2_6_hr
0    0.998932
1    0.001068
Name: proportion, dtype: float64

## Severe Class Imbalance and Overlap in the 2–6 Hour Derate Target

The target variable for predicting full derates within the next 2–6 hours is highly imbalanced. The majority of observations belong to the non-derate class (`0`), while only a very small proportion represent upcoming derate events (`1`).

This extreme imbalance presents a major machine learning challenge because models may become biased toward predicting the majority non-derate class. In addition, exploratory analysis revealed substantial operational overlap between derate and non-derate observations, particularly among highly frequent SPNs and lamp-status conditions that appeared commonly in both classes.

To address these challenges, multiple strategies were explored during model development, including chronological train-test splitting, near-term derate filtering, imbalance-aware ensemble methods, anomaly detection, and targeted SPN- and lamp-based feature engineering. Additional interaction-based feature engineering was also investigated to evaluate whether simultaneous high-risk fault and warning-state behavior improves near-term derate separability under chronological validation.

In [66]:
# Check Columns Containing Missing Values
X.isna().sum()[X.isna().sum() > 0]

min_since_last_fault    1063
dtype: int64

### Missing Value Assessment

The predictor dataset contains very few missing values relative to the total dataset size. Only the `min_since_last_fault` feature contains missing observations (1,064 rows), while all other inspected features contain complete values.

Because the missingness is limited to a small portion of the dataset, imputation techniques can be applied without substantially affecting the overall data distribution. This step is important because machine learning methods such as SMOTE and Random Forest modeling require complete numeric input data.

### Chronological Train-Test Split Experiment

In [67]:
# Chronological split without shuffling
split_index = int(len(X) * 0.7)

X_train_chrono = X.iloc[:split_index].copy()
X_test_chrono = X.iloc[split_index:].copy()

y_train_chrono = y.iloc[:split_index].copy()
y_test_chrono = y.iloc[split_index:].copy()

X_train_chrono.shape, X_test_chrono.shape, y_train_chrono.shape, y_test_chrono.shape

((554523, 86), (237653, 86), (554523,), (237653,))

In [68]:
# Check Target Class Counts in Chronological Training Set
y_train_chrono.value_counts()

derate_2_6_hr
0    553997
1       526
Name: count, dtype: int64

In [69]:
# Check Target Class Counts in Chronological Testing Set
y_test_chrono.value_counts()

derate_2_6_hr
0    237333
1       320
Name: count, dtype: int64

In [70]:
# Check Target Class Proportions in Chronological Training Set
y_train_chrono.value_counts(normalize=True)

derate_2_6_hr
0    0.999051
1    0.000949
Name: proportion, dtype: float64

In [71]:
# Check Target Class Proportions in Chronological Testing Set
y_test_chrono.value_counts(normalize=True)

derate_2_6_hr
0    0.998653
1    0.001347
Name: proportion, dtype: float64

In [72]:
# Check Missing Values in Chronological Training Features
X_train_chrono.isna().sum().sort_values(ascending=False).head(10)

min_since_last_fault     676
ESS_Id                     0
spn                        0
fmi                        0
active                     0
ecuSource                  0
MCTNumber                  0
Latitude                   0
Longitude                  0
num__AcceleratorPedal      0
dtype: int64

In [73]:
# Impute Missing Values Using Iterative Imputer
imputer = IterativeImputer(
    random_state=321,
    max_iter=10
)

X_train_chrono_imputed = imputer.fit_transform(X_train_chrono)
X_test_chrono_imputed = imputer.transform(X_test_chrono)

In [74]:
# # Check Type of Imputed Training Data
type(X_train_chrono_imputed)

numpy.ndarray

In [75]:
# Convert Imputed Arrays Back to DataFrames
X_train_chrono = pd.DataFrame(
    X_train_chrono_imputed,
    columns=X_train_chrono.columns,
    index=X_train_chrono.index
)

X_test_chrono = pd.DataFrame(
    X_test_chrono_imputed,
    columns=X_test_chrono.columns,
    index=X_test_chrono.index
)

## Stratified K-Fold Cross-Validation for Training Set Stability

Because the derate prediction target is highly imbalanced, Stratified K-Fold cross-validation was used to evaluate model stability within the training data before final chronological testing. Unlike regular K-Fold validation, Stratified K-Fold preserves the proportion of derate and non-derate observations across each fold, providing a more reliable assessment of model consistency under severe imbalance conditions.

This validation step helps determine whether the model learns stable predictive patterns across multiple training subsets prior to evaluation on the chronological holdout dataset.

In [76]:
# Import Cross-Validation and Balanced Random Forest Tools
from imblearn.ensemble import BalancedRandomForestClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

In [77]:
# Create Balanced Random Forest Model

brf_model = BalancedRandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

In [78]:
# Create Stratified K-Fold Validation Object

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=321
)

In [79]:
# Run Stratified K-Fold Cross-Validation
cv_scores = cross_val_score(
    brf_model,
    X_train_chrono,
    y_train_chrono,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)


In [80]:
# Display Cross-Validation F1 Scores
cv_scores

array([0.02549759, 0.02715529, 0.02221945, 0.0240481 , 0.02399899])

In [81]:
# Calculate Mean Cross-Validation F1 Score
cv_scores.mean()

np.float64(0.024583882671984043)

## Stratified K-Fold Validation Indicates Stable Learning but Persistent Class Overlap Challenges

The Stratified K-Fold cross-validation results demonstrated relatively stable F1 performance across all training folds, suggesting that the model learned consistent predictive patterns despite severe class imbalance and substantial overlap between derate and non-derate operational behavior. However, the overall F1 scores remained low, indicating that near-term derate prediction continues to be strongly limited by weak separability within the available feature space.

Cross-validation results demonstrated relatively stable model behavior across training folds, suggesting that poor minority-class performance was not primarily driven by severe overfitting. Instead, the consistently low F1 scores across folds indicate that substantial operational overlap and weak class separability remain the dominant challenges for near-term derate prediction.

In [82]:
# Rebuild 2-6 Hour Chronological Training and Testing Data

X_train_chrono.to_csv("X_2_6_train_chrono.csv", index=False)
X_test_chrono.to_csv("X_2_6_test_chrono.csv", index=False)


In [83]:
type(y_train_chrono), y_train_chrono.shape

(pandas.core.series.Series, (554523,))

In [84]:
y_train_chrono = y_train_chrono.squeeze()
y_test_chrono = y_test_chrono.squeeze()

In [85]:

# Save Target Variables as CSV Files

y_train_chrono.to_csv(
    "y_train_chrono.csv",
    index=False
)

y_test_chrono.to_csv(
    "y_test_chrono.csv",
    index=False
)

In [86]:
# Verify Missing Values After Imputation
X_train_chrono.isna().sum().sort_values(ascending=False).head(10)

ESS_Id                   0
ecuSource                0
spn                      0
fmi                      0
active                   0
activeTransitionCount    0
MCTNumber                0
Latitude                 0
Longitude                0
num__AcceleratorPedal    0
dtype: int64

In [87]:
# Apply SMOTE to the Chronological Training Set

smote = SMOTE(random_state=321)

X_train_chrono_smote, y_train_chrono_smote = smote.fit_resample(
    X_train_chrono,
    y_train_chrono
)

In [88]:
# Check Class Balance After SMOTE
y_train_chrono_smote.value_counts()

derate_2_6_hr
0    553997
1    553997
Name: count, dtype: int64

In [89]:
# Train Random Forest Model on SMOTE-Balanced Training Data
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

rf_model.fit(
    X_train_chrono_smote,
    y_train_chrono_smote
)
print("Model training complete")

Model training complete


In [90]:
# Examine Random Forest Feature Importance
feature_importance = pd.DataFrame({
    'feature': X_train_chrono_smote.columns,
    'importance': rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='importance',
    ascending=False
)

feature_importance.head(20)

,feature,importance
2,spn,0.117531
79,high_risk_lamp_flag,0.069199
77,spn_occurrence_count,0.063375
45,cat__LampStatus_18431,0.053599
36,cat__LampStatus_1023,0.039336
3,fmi,0.034623
1,ecuSource,0.034566
20,num__FuelLtd,0.034475
5,activeTransitionCount,0.033083
34,cat__IgnStatus_nan,0.027916


In [91]:
# Generate Prediction Probabilities and Class Predictions
y_prob = rf_model.predict_proba(X_test_chrono)[:, 1]

y_pred = (y_prob >= 0.2).astype(int)

In [92]:
from sklearn.metrics import classification_report, confusion_matrix
# Evaluate Random Forest Model Performance
print(confusion_matrix(y_test_chrono, y_pred))
print(classification_report(y_test_chrono, y_pred))

[[234071   3262]
 [   207    113]]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    237333
           1       0.03      0.35      0.06       320

    accuracy                           0.99    237653
   macro avg       0.52      0.67      0.53    237653
weighted avg       1.00      0.99      0.99    237653



In [93]:
# Train Balanced Random Forest Model

brf_model = BalancedRandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

brf_model.fit(
    X_train_chrono,
    y_train_chrono
)

BalancedRandomForestClassifier(max_depth=20, min_samples_leaf=5, n_jobs=-1,
                               random_state=321)

In [94]:
# Generate Balanced Random Forest Predictions

y_prob_brf = brf_model.predict_proba(X_test_chrono)[:, 1]

y_pred_brf = (y_prob_brf >= 0.7).astype(int)

In [95]:
# Evaluate Balanced Random Forest Model

print(confusion_matrix(y_test_chrono, y_pred_brf))
print(classification_report(y_test_chrono, y_pred_brf))

[[224905  12428]
 [   107    213]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97    237333
           1       0.02      0.67      0.03       320

    accuracy                           0.95    237653
   macro avg       0.51      0.81      0.50    237653
weighted avg       1.00      0.95      0.97    237653



## Stratified K-Fold Validation and Comparative Model Performance Under Severe Operational Overlap

Stratified K-Fold cross-validation was used to evaluate model stability within the training dataset before final chronological testing. Because the derate prediction targets were extremely imbalanced, regular K-Fold validation was not considered appropriate, since random folds could contain highly uneven distributions of derate and non-derate observations. Stratified K-Fold preserves class proportions across all folds, providing a more reliable assessment of model consistency under severe imbalance conditions.

Cross-validation results demonstrated relatively stable F1 performance across all folds, suggesting that poor minority-class performance was not primarily driven by severe overfitting. Instead, consistently low F1 scores indicated that substantial operational overlap and weak separability between derate and non-derate observations remained the dominant modeling challenges.

Comparative modeling experiments further demonstrated important behavioral differences between Standard Random Forest and Balanced Random Forest approaches. For the 2–6 hour prediction target, the Standard Random Forest model produced more controlled false-positive behavior while maintaining moderate near-term derate detection. In contrast, the Balanced Random Forest model substantially improved minority-class recall, indicating greater sensitivity toward upcoming derate events, but this improvement was accompanied by a large increase in false-positive predictions due to persistent overlap between operational states.

Additional exploratory analysis revealed that several highly frequent SPNs and lamp-status conditions appeared commonly in both derate and non-derate classes. Targeted feature engineering using high-risk SPN indicators, high-risk lamp-status indicators, and interaction-based features modestly improved separability and reduced false-positive behavior in some experiments. However, overall predictive performance remained constrained by substantial overlap within the available operational feature space.

Overall, the experiments demonstrated that near-term derate prediction is highly challenging under realistic chronological validation conditions. The findings suggest that warning signals become stronger closer to the derate event itself, while earlier prediction windows exhibit weaker and more ambiguous operational patterns. The project further highlights the importance of chronological validation, imbalance-aware learning, and interpretable feature engineering when developing predictive maintenance models for highly overlapping real-world operational data.

Near-term derate prediction may require richer sequential or temporal behavioral features rather than isolated snapshot operational variables alone.

# 6–12 Hour Full Derate Prediction Using Chronological Split

After evaluating the 2–6 hour prediction window, the analysis was extended to the 6–12 hour derate prediction target. This experiment investigates whether operational and fault-related patterns are more detectable at an earlier stage before full derate occurs.

The same chronological train-test split workflow was used to preserve temporal order and better simulate a real-world predictive maintenance setting. The modeling pipeline includes missing value imputation, SMOTE oversampling on the training data, Random Forest classification, and threshold-based evaluation.

In [96]:
# Define Prediction Target and Predictor Variables

# Set the target column
y_6_12 = model_data_filtered['derate_6_12_hr']

# Define predictor features
X_6_12 = model_data_filtered.drop(
    columns=[
        'derate_2_6_hr',
        'derate_6_12_hr',
        'derate_12_24_hr',
        'hours_to_derate',
        'next_derate_time',
        'next_derate_time_x',
        'next_derate_time_y',
        'EventTimeStamp',
        'LocationTimeStamp',
        'RecordID',
        'FaultId',
        'eventDescription',
        'ecuSoftwareVersion',
        'ecuSerialNumber',
        'ecuModel',
        'ecuMake',
        'EquipmentID',
        'actionDescription',
        'faultValue'
    ],
    errors='ignore'
)

In [97]:
# Verify Predictor and Target Shapes

X_6_12.shape, y_6_12.shape

((792176, 86), (792176,))

In [98]:
# Create Chronological Train-Test Split for the 6–12 Hour Prediction Model
split_index = int(len(X_6_12) * 0.7)

X_6_12_train_chrono = X_6_12.iloc[:split_index].copy()
X_6_12_test_chrono = X_6_12.iloc[split_index:].copy()

y_6_12_train_chrono = y_6_12.iloc[:split_index].copy()
y_6_12_test_chrono = y_6_12.iloc[split_index:].copy()

In [99]:
# Verify Shapes of Chronological Training and Testing Sets
X_6_12_train_chrono.shape, X_6_12_test_chrono.shape, y_6_12_train_chrono.shape, y_6_12_test_chrono.shape

((554523, 86), (237653, 86), (554523,), (237653,))

In [100]:
# Check Target Class Counts in the Chronological Training Set
y_6_12_train_chrono.value_counts()

derate_6_12_hr
0    554036
1       487
Name: count, dtype: int64

In [101]:
# Check Target Class Counts in the Chronological Testing Set
y_6_12_test_chrono.value_counts()

derate_6_12_hr
0    237422
1       231
Name: count, dtype: int64

In [102]:
# Check Target Class Proportions in the Chronological Training Set
y_6_12_train_chrono.value_counts(normalize=True)

derate_6_12_hr
0    0.999122
1    0.000878
Name: proportion, dtype: float64

In [103]:
# Check Target Class Proportions in the Chronological Testing Set
y_6_12_test_chrono.value_counts(normalize=True)

derate_6_12_hr
0    0.999028
1    0.000972
Name: proportion, dtype: float64

In [104]:
# Check Missing Values in Chronological Training Features
X_6_12_train_chrono.isna().sum().sort_values(ascending=False).head(10)

min_since_last_fault     676
ESS_Id                     0
spn                        0
fmi                        0
active                     0
ecuSource                  0
MCTNumber                  0
Latitude                   0
Longitude                  0
num__AcceleratorPedal      0
dtype: int64

In [105]:
# Check Missing Values in Chronological Testing Features
X_6_12_test_chrono.isna().sum().sort_values(ascending=False).head(10)

min_since_last_fault     387
ESS_Id                     0
spn                        0
fmi                        0
active                     0
ecuSource                  0
MCTNumber                  0
Latitude                   0
Longitude                  0
num__AcceleratorPedal      0
dtype: int64

In [106]:
# Impute Missing Values Using Iterative Imputer
imputer = IterativeImputer(
    random_state=321,
    max_iter=10
)

X_6_12_train_chrono_imputed = imputer.fit_transform(X_6_12_train_chrono)

X_6_12_test_chrono_imputed = imputer.transform(X_6_12_test_chrono)

In [107]:
# Convert Imputed Arrays Back to DataFrames
X_6_12_train_chrono = pd.DataFrame(
    X_6_12_train_chrono_imputed,
    columns=X_6_12_train_chrono.columns,
    index=X_6_12_train_chrono.index
)

X_6_12_test_chrono = pd.DataFrame(
    X_6_12_test_chrono_imputed,
    columns=X_6_12_test_chrono.columns,
    index=X_6_12_test_chrono.index
)

In [108]:
# Run Stratified K-Fold Validation for 6–12 Hour Training Set

brf_model_6_12 = BalancedRandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

In [109]:
# Create Stratified K-Fold Validation Object

cv_scores_6_12 = cross_val_score(
    brf_model_6_12,
    X_6_12_train_chrono,
    y_6_12_train_chrono,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)



In [110]:
# Display Cross-Validation F1 Scores
cv_scores_6_12

array([0.0310192 , 0.03556444, 0.03066378, 0.02917046, 0.03071919])

In [111]:
# Calculate Mean Cross-Validation F1 Score
cv_scores_6_12.mean()

np.float64(0.03142741479096985)

The 6–12 hour target demonstrated relatively stable cross-validation behavior across training folds, suggesting that the model learned consistent operational patterns within the training dataset. However, chronological holdout evaluation still produced weaker minority-class performance compared with the 2–6 hour target, indicating that earlier derate warning behavior remains more difficult to generalize under realistic future operational conditions.

In [112]:
# Rebuild 6_12 Hour Chronological Training and Testing Data


X_6_12_train_chrono.to_csv("X_6_12_train_chrono.csv", index=False)
X_6_12_test_chrono.to_csv("X_6_12_test_chrono.csv", index=False)
y_6_12_train_chrono.to_csv("y_6_12_train_chrono.csv", index=False)
y_6_12_test_chrono.to_csv("y_6_12_test_chrono.csv", index=False)


In [113]:
# Verify Missing Values After Imputation
X_6_12_train_chrono.isna().sum().sort_values(ascending=False).head(10)

ESS_Id                   0
ecuSource                0
spn                      0
fmi                      0
active                   0
activeTransitionCount    0
MCTNumber                0
Latitude                 0
Longitude                0
num__AcceleratorPedal    0
dtype: int64

In [114]:
X_6_12_test_chrono.isna().sum().sort_values(ascending=False).head(10)

ESS_Id                   0
ecuSource                0
spn                      0
fmi                      0
active                   0
activeTransitionCount    0
MCTNumber                0
Latitude                 0
Longitude                0
num__AcceleratorPedal    0
dtype: int64

In [115]:
# Apply SMOTE to the Chronological Training Set

smote = SMOTE(random_state=321)

X_6_12_train_chrono_smote, y_6_12_train_chrono_smote = smote.fit_resample(
    X_6_12_train_chrono,
    y_6_12_train_chrono
)

In [116]:
# Check Class Balance After SMOTE

y_6_12_train_chrono_smote.value_counts()

derate_6_12_hr
0    554036
1    554036
Name: count, dtype: int64

In [117]:
# Train Random Forest Model for the 6–12 Hour Derate Prediction

rf_model_6_12 = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

rf_model_6_12.fit(
    X_6_12_train_chrono_smote,
    y_6_12_train_chrono_smote
)

RandomForestClassifier(max_depth=20, min_samples_leaf=5, n_jobs=-1,
                       random_state=321)

In [118]:
# Generate Prediction Probabilities and Class Predictions

y_prob_6_12 = rf_model_6_12.predict_proba(
    X_6_12_test_chrono
)[:, 1]

y_pred_6_12 = (y_prob_6_12 >= 0.1).astype(int)

In [119]:
pd.Series(y_pred_6_12).value_counts()

0    234627
1      3026
Name: count, dtype: int64

In [120]:
print(confusion_matrix(y_6_12_test_chrono, y_pred_6_12))
print(classification_report(y_6_12_test_chrono, y_pred_6_12))

[[234452   2970]
 [   175     56]]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    237422
           1       0.02      0.24      0.03       231

    accuracy                           0.99    237653
   macro avg       0.51      0.61      0.51    237653
weighted avg       1.00      0.99      0.99    237653



## Impact of Validation Strategy on Prediction Performance

The choice of validation strategy significantly affected model performance across prediction windows. Under random stratified splitting, the 6–12 hour prediction window appeared to outperform the 2–6 hour window. However, under chronological validation, the 2–6 hour window produced stronger predictive performance.

This contrast suggests that random splitting may overestimate performance for earlier prediction horizons by mixing similar temporal patterns between training and testing data. Chronological splitting provides a more realistic evaluation of future derate prediction and indicates that the strongest predictive signals occur closer to the derate event itself.

## Balanced Random Forest Experiment

The standard Random Forest model showed limited performance under chronological validation, particularly for the minority derate class. Although SMOTE improved class balance during training, the model still struggled to clearly separate derate and non-derate events, especially at practical probability thresholds.

To further address the extreme class imbalance, a Balanced Random Forest model was tested. Unlike standard Random Forest, Balanced Random Forest automatically balances the classes during the construction of each decision tree by sampling minority and majority classes more evenly. This approach reduces the model’s tendency to favor the majority non-derate class without relying on synthetic oversampling.

This experiment investigates whether imbalance-aware tree sampling can improve the detection of future derate events under a chronological train-test split while maintaining the overall Random Forest framework used throughout the project.

In [121]:
# Train Balanced Random Forest Model

from imblearn.ensemble import BalancedRandomForestClassifier

brf_model_6_12 = BalancedRandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

brf_model_6_12.fit(
    X_6_12_train_chrono,
    y_6_12_train_chrono
)

BalancedRandomForestClassifier(max_depth=20, min_samples_leaf=5, n_jobs=-1,
                               random_state=321)

In [122]:
# Generate Balanced Random Forest Prediction Probabilities and Class Predictions

y_prob_6_12 = brf_model_6_12.predict_proba(
    X_6_12_test_chrono
)[:, 1]

y_pred_6_12 = (y_prob_6_12 >= 0.7).astype(int)

In [123]:
# Evaluate Balanced Random Forest Model Performance

print(confusion_matrix(y_6_12_test_chrono, y_pred_6_12))
print(classification_report(y_6_12_test_chrono, y_pred_6_12))

[[234936   2486]
 [   165     66]]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    237422
           1       0.03      0.29      0.05       231

    accuracy                           0.99    237653
   macro avg       0.51      0.64      0.52    237653
weighted avg       1.00      0.99      0.99    237653




## Overall Summary of Model Performance and Prediction Challenges

The modeling experiments demonstrated meaningful differences in model behavior across the 2–6 hour and 6–12 hour prediction windows under chronological validation. The standard Random Forest model performed relatively better for the 2–6 hour target, suggesting that warning signals become stronger and more distinguishable closer to the derate event. In contrast, the Balanced Random Forest model produced more stable minority-class detection for the 6–12 hour target, indicating that earlier warning patterns may be weaker and require imbalance-aware learning strategies.

Additional investigation revealed that long-term future derate observations introduced substantial overlap between derate and non-derate classes. Filtering the dataset to focus on observations within 24 hours of future derate events improved label relevance and reduced some overlap-related noise.

However, despite differences between models and prediction windows, overall predictive performance remained limited due to the continued dominance of false-positive predictions. This suggests that substantial overlap still exists between derate and non-derate operational behavior within the available feature space, making near-term derate prediction highly challenging under realistic chronological validation.

## 12–24 Hour Prediction Window Analysis

The final modeling phase focused on predicting full derate events occurring within the 12–24 hour prediction window. Compared with the shorter prediction horizons, this target is more challenging because warning signals are weaker and class overlap increases farther from the derate event.

As in the previous analyses, chronological train-test splitting was used to maintain realistic temporal validation and avoid information leakage. Multiple models were evaluated to determine whether early derate signals could still be detected effectively within the available feature space.

In [124]:
# Define predictor variables and target label

X_12_24 = predictive_variable
y_12_24 = model_data_filtered['derate_12_24_hr']

In [125]:
# Create Chronological Train-Test Split for the 12–24 Hour Prediction Model
split_index = int(len(X_12_24) * 0.7)

X_12_24_train_chrono = X_12_24.iloc[:split_index].copy()
X_12_24_test_chrono = X_12_24.iloc[split_index:].copy()

y_12_24_train_chrono = y_12_24.iloc[:split_index].copy()
y_12_24_test_chrono = y_12_24.iloc[split_index:].copy()

In [126]:
# Verify Shapes of Chronological Training and Testing Sets
X_12_24_train_chrono.shape, X_12_24_test_chrono.shape, y_12_24_train_chrono.shape, y_12_24_test_chrono.shape

((554523, 86), (237653, 86), (554523,), (237653,))

In [127]:
### Missing Value Inspection and Imputation

X_12_24_train_chrono.isna().sum().sort_values(ascending=False).head(15)

min_since_last_fault             676
ESS_Id                             0
spn                                0
fmi                                0
active                             0
ecuSource                          0
MCTNumber                          0
Latitude                           0
Longitude                          0
num__AcceleratorPedal              0
num__BarometricPressure            0
num__CruiseControlSetSpeed         0
num__DistanceLtd                   0
num__EngineCoolantTemperature      0
num__EngineLoad                    0
dtype: int64

In [128]:
X_12_24_test_chrono.isna().sum().sort_values(ascending=False).head(15)

min_since_last_fault             387
ESS_Id                             0
spn                                0
fmi                                0
active                             0
ecuSource                          0
MCTNumber                          0
Latitude                           0
Longitude                          0
num__AcceleratorPedal              0
num__BarometricPressure            0
num__CruiseControlSetSpeed         0
num__DistanceLtd                   0
num__EngineCoolantTemperature      0
num__EngineLoad                    0
dtype: int64

In [129]:
# Impute Missing Values Using Iterative Imputer
imputer = IterativeImputer(
    random_state=321,
    max_iter=10
)

X_12_24_train_chrono_imputed = imputer.fit_transform(X_12_24_train_chrono)

X_12_24_test_chrono_imputed = imputer.transform(X_12_24_test_chrono)


In [130]:
# Convert Imputed Arrays Back to DataFrames
X_12_24_train_chrono = pd.DataFrame(
    X_12_24_train_chrono_imputed,
    columns=X_12_24_train_chrono.columns,
    index=X_12_24_train_chrono.index
)

X_12_24_test_chrono = pd.DataFrame(
    X_12_24_test_chrono_imputed,
    columns=X_12_24_test_chrono.columns,
    index=X_12_24_test_chrono.index
)

In [131]:
# Verify Remaining Missing Values After Imputation

X_12_24_train_chrono.isna().sum().sort_values(
    ascending=False
).head(10)

ESS_Id                   0
ecuSource                0
spn                      0
fmi                      0
active                   0
activeTransitionCount    0
MCTNumber                0
Latitude                 0
Longitude                0
num__AcceleratorPedal    0
dtype: int64

In [132]:
X_12_24_test_chrono.isna().sum().sort_values(
    ascending=False
).head(10)

ESS_Id                   0
ecuSource                0
spn                      0
fmi                      0
active                   0
activeTransitionCount    0
MCTNumber                0
Latitude                 0
Longitude                0
num__AcceleratorPedal    0
dtype: int64

In [133]:
# Run Stratified K-Fold Validation for 6–12 Hour Training Set

brf_model_12_24 = BalancedRandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

In [134]:
# Create Stratified K-Fold Validation Object

cv_scores_12_24 = cross_val_score(
    brf_model_12_24,
    X_12_24_train_chrono,
    y_12_24_train_chrono,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)



In [135]:
# Display Cross-Validation F1 Scores
cv_scores_12_24

array([0.03699565, 0.03440556, 0.03380079, 0.0350401 , 0.0340757 ])

In [136]:
# Display Cross-Validation F1 Scores
cv_scores_12_24.mean()

np.float64(0.03486356056656313)

In [137]:
# Save 12–24 Hour Training and Testing Datasets as CSV Files
X_12_24_train_chrono.to_csv(
    "X_12_24_train_chrono.csv",
    index=False
)

X_12_24_test_chrono.to_csv(
    "X_12_24_test_chrono.csv",
    index=False
)

y_12_24_train_chrono.to_csv(
    "y_12_24_train_chrono.csv",
    index=False
)

y_12_24_test_chrono.to_csv(
    "y_12_24_test_chrono.csv",
    index=False
)


In [138]:
# Apply SMOTE to the 12–24 Hour Chronological Training Set

smote = SMOTE(random_state=321)

X_12_24_train_chrono_smote, y_12_24_train_chrono_smote = smote.fit_resample(
    X_12_24_train_chrono,
    y_12_24_train_chrono
)

In [139]:
# Check Class Balance After SMOTE

y_12_24_train_chrono_smote.value_counts()



derate_12_24_hr
0    553702
1    553702
Name: count, dtype: int64

In [140]:
# Train Standard Random Forest Model on SMOTE-Balanced 12–24 Hour Training Data

rf_model_12_24 = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

rf_model_12_24.fit(
    X_12_24_train_chrono_smote,
    y_12_24_train_chrono_smote
)

RandomForestClassifier(max_depth=20, min_samples_leaf=5, n_jobs=-1,
                       random_state=321)

In [141]:
# Generate Standard Random Forest Prediction Probabilities

y_prob_12_24 = rf_model_12_24.predict_proba(
    X_12_24_test_chrono
)[:, 1]

In [142]:
# Apply Probability Threshold to Generate Final Predictions

y_pred_12_24 = (
    y_prob_12_24 >= 0.2
).astype(int)

In [161]:
# Verify Standard Random Forest Prediction Shapes
y_12_24_test_chrono.shape, y_prob_12_24.shape, y_pred_12_24.shape

((237653,), (237653,), (237653,))

In [143]:
# Evaluate Standard Random Forest Model Performance

print(confusion_matrix(
    y_12_24_test_chrono,
    y_pred_12_24
))

print(classification_report(
    y_12_24_test_chrono,
    y_pred_12_24
))

[[236830    570]
 [   244      9]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    237400
           1       0.02      0.04      0.02       253

    accuracy                           1.00    237653
   macro avg       0.51      0.52      0.51    237653
weighted avg       1.00      1.00      1.00    237653



## Standard Random Forest Performance for the 12–24 Hour Prediction Target

The Standard Random Forest model produced substantially weaker minority-class performance for the 12–24 hour derate prediction target compared with the shorter prediction windows. Although overall accuracy remained high due to extreme class imbalance, the model detected very few true derate events and demonstrated very low minority-class recall and F1 performance.

These results suggest that operational behavior occurring 12–24 hours before a full derate remains highly difficult to distinguish from normal non-derate operating conditions. As the prediction horizon increases, overlap between derate and non-derate observations becomes increasingly dominant, substantially reducing model separability under chronological validation.

Because the Standard Random Forest model remained highly conservative and struggled to detect minority-class derate observations, a Balanced Random Forest approach was further evaluated. Balanced Random Forest internally balances majority and minority classes during tree construction, potentially improving sensitivity toward rare derate events under severe imbalance conditions.

## Relationship Between Cross-Validation and Chronological Test Evaluation

The cross-validation F1 scores were consistently higher than the final chronological test F1 scores, indicating that the model learned some repeatable operational patterns within the training dataset. However, the lower chronological test performance suggests that these learned patterns generalized poorly to future unseen operational conditions. This performance gap further supports the presence of substantial operational overlap and weak separability between derate and non-derate observations across time.


In [144]:
# Train Balanced Random Forest Model for the 12–24 Hour Prediction Target

brf_model_12_24 = BalancedRandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

brf_model_12_24.fit(
    X_12_24_train_chrono,
    y_12_24_train_chrono
)

BalancedRandomForestClassifier(max_depth=20, min_samples_leaf=5, n_jobs=-1,
                               random_state=321)

In [149]:
y_12_24_test_chrono = y_12_24_test_chrono.squeeze()

In [151]:
type(y_12_24_test_chrono), y_12_24_test_chrono.shape

(pandas.core.series.Series, (237653,))

In [152]:
y_pred_brf_12_24 = y_pred_brf_12_24.ravel()

In [157]:
# Generate Balanced Random Forest Prediction Probabilities
y_prob_brf_12_24 = brf_model_12_24.predict_proba(
    X_12_24_test_chrono
)[:, 1]

In [158]:
# Apply Probability Threshold to Generate Final Balanced Random Forest Predictions
y_pred_brf_12_24 = (
    y_prob_brf_12_24 >= 0.7
).astype(int)

In [159]:
# Verify Matching Shapes Between Test Targets, Prediction Probabilities, and Final Predictions
y_12_24_test_chrono.shape, y_prob_brf_12_24.shape, y_pred_brf_12_24.shape

((237653,), (237653,), (237653,))

In [160]:
# Evaluate Balanced Random Forest Model Performance
print(confusion_matrix(
    y_12_24_test_chrono,
    y_pred_brf_12_24
))

print(classification_report(
    y_12_24_test_chrono,
    y_pred_brf_12_24
))

[[234895   2505]
 [   194     59]]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    237400
           1       0.02      0.23      0.04       253

    accuracy                           0.99    237653
   macro avg       0.51      0.61      0.52    237653
weighted avg       1.00      0.99      0.99    237653



## 12–24 Hour Prediction Target: Comparative Model Performance and Validation Interpretation

The 12–24 hour derate prediction target produced the weakest overall minority-class performance among all evaluated prediction windows, indicating that early operational warning behavior remains highly difficult to distinguish from normal non-derate conditions. As the prediction horizon moved farther away from the full derate event, substantial overlap between derate and non-derate observations increasingly limited model separability under realistic chronological validation conditions.

Comparative modeling experiments revealed a substantial behavioral difference between the Standard Random Forest and Balanced Random Forest approaches for this target window. The Standard Random Forest model, even after SMOTE preprocessing, remained highly conservative and detected very few true derate events. In contrast, the Balanced Random Forest model substantially improved minority-class recall and F1 performance by internally balancing majority and minority classes during ensemble tree construction.

These findings suggest that aggressive imbalance-aware learning became increasingly important as the predictive signal weakened farther from the derate event itself. Under severe overlap conditions, SMOTE-based preprocessing alone was insufficient to meaningfully improve minority-class sensitivity, whereas Balanced Random Forest more effectively forced the ensemble to learn rare derate-related operational patterns.

Stratified K-Fold cross-validation further demonstrated relatively stable model behavior across training folds. Importantly, the cross-validation F1 scores and final chronological test F1 scores remained relatively similar, indicating that severe overfitting was not the dominant modeling limitation. Instead, both validation approaches consistently produced low but stable minority-class performance, suggesting that the primary challenge was weak separability and substantial operational overlap within the available feature space.

Overall, the 12–24 hour experiments demonstrated that future derate prediction becomes increasingly difficult as operational behavior moves farther away from the actual derate event. While imbalance-aware ensemble learning improved minority-class sensitivity, chronological evaluation confirmed that early-warning operational patterns remained subtle, ambiguous, and difficult to generalize under realistic future deployment conditions.

## Final Model Comparison and Net Savings Evaluation

After evaluating Standard Random Forest and Balanced Random Forest models across the three derate prediction windows, a final comparison table was created to summarize classification performance, operational outcomes, and estimated business savings.

Because the primary project objective is reducing operational losses associated with full derates, the final evaluation emphasizes estimated net savings rather than classification metrics alone. In this analysis, each correctly predicted full derate is assumed to prevent approximately `$4,000` in towing and repair costs, while each false positive prediction is assumed to generate approximately `$500` in unnecessary maintenance and downtime expenses.

This comparison helps identify which target window and modeling strategy provides the greatest potential operational value under realistic deployment conditions.

In [167]:
# Create Final Model Comparison and Net Savings Summary Table
model_comparison = pd.DataFrame({

    'Target_Window': [
        '2–6 hr', '2–6 hr',
        '6–12 hr', '6–12 hr',
        '12–24 hr', '12–24 hr'
    ],

    'Model': [
        'Standard RF + SMOTE',
        'Balanced RF',
        'Standard RF + SMOTE',
        'Balanced RF',
        'Standard RF + SMOTE',
        'Balanced RF'
    ],

    'Cross_Validation_F1': [
        0.0246, 0.0246,
        0.0314, 0.0314,
        0.0349, 0.0349
    ],

    'Test_F1': [
        0.06, 0.03,
        0.03, 0.05,
        0.02, 0.04
    ],

    'Precision': [
        0.03, 0.02,
        0.02, 0.03,
        0.02, 0.02
    ],

    'Recall': [
        0.35, 0.67,
        0.24, 0.29,
        0.04, 0.23
    ],

    'Correct_Derates_TP': [
        113, 213,
        56, 66,
        9, 59
    ],

    'False_Derates_FP': [
        3262, 12428,
        2970, 2486,
        570, 2505
    ],

    'Missed_Derates_FN': [
        207, 107,
        175, 165,
        244, 194
    ],

    'Correct_Non_Derates_TN': [
        234071, 224905,
        234452, 234936,
        236830, 234895
    ]

})

# Calculate Estimated Net Savings
model_comparison['Estimated_Net_Savings'] = (
    model_comparison['Correct_Derates_TP'] * 4000
) - (
    model_comparison['False_Derates_FP'] * 500
)

# Sort by Highest Net Savings
model_comparison = model_comparison.sort_values(
    by='Estimated_Net_Savings',
    ascending=False
)

model_comparison

,Target_Window,Model,Cross_Validation_F1,Test_F1,Precision,Recall,Correct_Derates_TP,False_Derates_FP,Missed_Derates_FN,Correct_Non_Derates_TN,Estimated_Net_Savings
4,12–24 hr,Standard RF + SMOTE,0.0349,0.02,0.02,0.04,9,570,244,236830,-249000
3,6–12 hr,Balanced RF,0.0314,0.05,0.03,0.29,66,2486,165,234936,-979000
5,12–24 hr,Balanced RF,0.0349,0.04,0.02,0.23,59,2505,194,234895,-1016500
0,2–6 hr,Standard RF + SMOTE,0.0246,0.06,0.03,0.35,113,3262,207,234071,-1179000
2,6–12 hr,Standard RF + SMOTE,0.0314,0.03,0.02,0.24,56,2970,175,234452,-1261000
1,2–6 hr,Balanced RF,0.0246,0.03,0.02,0.67,213,12428,107,224905,-5362000


In [168]:
model_comparison = model_comparison.sort_values(
    by='Estimated_Net_Savings',
    ascending=False
).reset_index(drop=True)

In [169]:
model_comparison

,Target_Window,Model,Cross_Validation_F1,Test_F1,Precision,Recall,Correct_Derates_TP,False_Derates_FP,Missed_Derates_FN,Correct_Non_Derates_TN,Estimated_Net_Savings
0,12–24 hr,Standard RF + SMOTE,0.0349,0.02,0.02,0.04,9,570,244,236830,-249000
1,6–12 hr,Balanced RF,0.0314,0.05,0.03,0.29,66,2486,165,234936,-979000
2,12–24 hr,Balanced RF,0.0349,0.04,0.02,0.23,59,2505,194,234895,-1016500
3,2–6 hr,Standard RF + SMOTE,0.0246,0.06,0.03,0.35,113,3262,207,234071,-1179000
4,6–12 hr,Standard RF + SMOTE,0.0314,0.03,0.02,0.24,56,2970,175,234452,-1261000
5,2–6 hr,Balanced RF,0.0246,0.03,0.02,0.67,213,12428,107,224905,-5362000


## Overall Comparative Summary and Operational Interpretation

This project evaluated multiple machine learning strategies for predicting future full derates using J1939 fault code and onboard vehicle diagnostic data across three prediction horizons: 2–6 hours, 6–12 hours, and 12–24 hours before a full derate event. Standard Random Forest models trained with SMOTE-balanced data and Balanced Random Forest models were compared using chronological validation, Stratified K-Fold cross-validation, operational outcomes, and estimated business savings.

Across all prediction windows, the models consistently demonstrated that predicting future full derates is a highly challenging classification problem due to severe class imbalance, substantial operational overlap, and weak separability between derate and non-derate operating conditions. As the prediction horizon moved farther away from the actual derate event, the predictive signal weakened further, making early-warning operational patterns increasingly difficult to distinguish from normal truck behavior.

Importantly, the relatively similar cross-validation and chronological test F1 scores suggested that severe overfitting was not the primary limitation of the modeling pipeline. Instead, the consistently low but stable minority-class performance across folds and future chronological data indicated that the underlying operational behavior itself was inherently difficult to separate using the available feature space. These findings suggest that the models learned limited but reasonably stable predictive structure under realistic future deployment conditions.

Comparative model evaluation further demonstrated that Balanced Random Forest models generally improved minority-class recall by more aggressively learning rare derate-related operational patterns. However, this increased sensitivity often produced substantially larger numbers of false positive predictions, reducing estimated operational savings under the assumed business cost structure. Although several models successfully identified upcoming full derates, the associated false positive maintenance burden frequently outweighed the operational savings generated by prevented towing and repair events.

Overall, these findings suggest that the current feature space and operational data provide only limited predictive signal for early full derate detection. Rather than indicating model failure, the results more likely reflect the inherent complexity and ambiguity of real-world heavy-duty vehicle fault behavior. Additional operational context, richer temporal feature engineering, sequential modeling approaches, or domain-specific maintenance information may be required to improve future predictive performance and economic deployment value.

Based on the combined evaluation of cross-validation stability, chronological test performance, recall behavior, and operational savings analysis, the strongest-performing candidate model will be selected for the final deployment-style evaluation using pre-2019 training data and isolated 2019 operational data as a fully unseen future test period.

In [170]:
model_data.columns

Index(['RecordID', 'ESS_Id', 'EventTimeStamp', 'eventDescription',
       'actionDescription', 'ecuSoftwareVersion', 'ecuSerialNumber',
       'ecuModel', 'ecuMake', 'ecuSource', 'spn', 'fmi', 'active',
       'activeTransitionCount', 'faultValue', 'EquipmentID', 'MCTNumber',
       'Latitude', 'Longitude', 'LocationTimeStamp', 'num__AcceleratorPedal',
       'num__BarometricPressure', 'num__CruiseControlSetSpeed',
       'num__DistanceLtd', 'num__EngineCoolantTemperature', 'num__EngineLoad',
       'num__EngineOilPressure', 'num__EngineOilTemperature', 'num__EngineRpm',
       'num__EngineTimeLtd', 'num__FuelLevel', 'num__FuelLtd', 'num__FuelRate',
       'num__FuelTemperature', 'num__IntakeManifoldTemperature',
       'num__ServiceDistance', 'num__Speed', 'num__SwitchedBatteryVoltage',
       'num__Throttle', 'num__TurboBoostPressure',
       'cat__CruiseControlActive_False', 'cat__CruiseControlActive_True',
       'cat__CruiseControlActive_nan', 'cat__IgnStatus_False',
       'cat__

## Final Deployment Evaluation Using Pre-2019 Training Data and 2019 Future Test Data

In [171]:
# Convert Event Timestamp to Datetime Format
model_data['EventTimeStamp'] = pd.to_datetime(
    model_data['EventTimeStamp']
)

In [172]:
# Create Event Year Variable
model_data['event_year'] = (
    model_data['EventTimeStamp'].dt.year
)

In [173]:
# Verify Available Operational Years
model_data['event_year'].value_counts().sort_index()

event_year
2000       204
2009         8
2010        22
2011       150
2015    294412
2016    294742
2017    226939
2018    122164
2019     95819
2020     14890
Name: count, dtype: int64

### Note:
A small number of operational records from years prior to 2015 were observed and may represent legacy or anomalous timestamp behavior. Additional discussion regarding potential temporal filtering may be warranted during future modeling refinement. However, for the current deployment-style evaluation, all available pre-2019 operational records were retained as training data in order to remain consistent with the project requirements and team evaluation approach.

In [179]:
# Create Final Derate Prediction Target Labels
model_data['derate_2_6_hr'] = (
    (model_data['hours_to_derate'] >= 2) &
    (model_data['hours_to_derate'] <= 6)
).astype(int)

model_data['derate_6_12_hr'] = (
    (model_data['hours_to_derate'] > 6) &
    (model_data['hours_to_derate'] <= 12)
).astype(int)

model_data['derate_12_24_hr'] = (
    (model_data['hours_to_derate'] > 12) &
    (model_data['hours_to_derate'] <= 24)
).astype(int)

In [180]:
# Split Data into Pre-2019 Training Data and 2019 Future Test Data
train_pre_2019 = model_data[
    model_data['event_year'] < 2019
].copy()

test_2019 = model_data[
    model_data['event_year'] == 2019
].copy()

train_pre_2019.shape, test_2019.shape

((938641, 96), (95819, 96))

## Rationale for Selecting the 2–6 Hour Prediction Target for Final Deployment Evaluation

Among the three evaluated prediction windows (2–6 hr, 6–12 hr, and 12–24 hr), the 2–6 hour target consistently demonstrated the strongest overall predictive performance across both Standard Random Forest and Balanced Random Forest modeling approaches. This target window generally produced higher recall, stronger F1 performance, and improved derate sensitivity relative to the longer prediction horizons.

As the prediction horizon moved farther away from the full derate event, substantial operational overlap and weaker early-warning signal behavior increasingly reduced model separability and predictive stability. Although none of the evaluated models produced positive net operational savings under the assumed deployment cost structure, the 2–6 hour prediction window provided the most stable and operationally informative balance between derate detection capability and false positive behavior.

Therefore, the 2–6 hour target was selected for the final deployment-style evaluation using pre-2019 training data and the isolated 2019 operational dataset as a fully unseen future test period.

In [182]:
# Define Final 2–6 Hour Full Derate Prediction Target
historical_derates = train_pre_2019['derate_2_6_hr']

deployment_2019_derates = test_2019['derate_2_6_hr']

In [184]:
# Create Final Deployment Feature Matrices and Remove Leakage Variables
historical_features = train_pre_2019.drop(
    columns=[
        'derate_2_6_hr',
        'derate_6_12_hr',
        'derate_12_24_hr',
        'hours_to_derate',
        'next_derate_time',
        'EventTimeStamp',
        'LocationTimeStamp',
        'RecordID',
        'FaultId',
        'eventDescription',
        'ecuSoftwareVersion',
        'ecuSerialNumber',
        'ecuModel',
        'ecuMake',
        'EquipmentID',
        'actionDescription',
        'faultValue'
    ],
    errors='ignore'
)

deployment_2019_features = test_2019.drop(
    columns=[
        'derate_2_6_hr',
        'derate_6_12_hr',
        'derate_12_24_hr',
        'hours_to_derate',
        'next_derate_time',
        'EventTimeStamp',
        'LocationTimeStamp',
        'RecordID',
        'FaultId',
        'eventDescription',
        'ecuSoftwareVersion',
        'ecuSerialNumber',
        'ecuModel',
        'ecuMake',
        'EquipmentID',
        'actionDescription',
        'faultValue'
    ],
    errors='ignore'
)

historical_features.shape, deployment_2019_features.shape

((938641, 79), (95819, 79))

In [185]:
# Check Missing Values Before Imputation
historical_features.isna().sum().sort_values(
    ascending=False
).head(15)

ESS_Id                           0
ecuSource                        0
spn                              0
fmi                              0
active                           0
activeTransitionCount            0
MCTNumber                        0
Latitude                         0
Longitude                        0
num__AcceleratorPedal            0
num__BarometricPressure          0
num__CruiseControlSetSpeed       0
num__DistanceLtd                 0
num__EngineCoolantTemperature    0
num__EngineLoad                  0
dtype: int64

In [186]:
# Verify Total Missing Values in 2019 Deployment Features
deployment_2019_features.isna().sum().sum()

np.int64(0)

In [187]:
# Check Class Distribution Before Applying SMOTE
historical_derates.value_counts()

derate_2_6_hr
0    937945
1       696
Name: count, dtype: int64

In [188]:
# Apply SMOTE to Historical Training Data Only
smote = SMOTE(
    random_state=321
)

historical_features_smote, historical_derates_smote = smote.fit_resample(
    historical_features,
    historical_derates
)

In [189]:
# Verify Balanced Class Distribution After SMOTE
historical_derates_smote.value_counts()

derate_2_6_hr
0    937945
1    937945
Name: count, dtype: int64

In [191]:
# Train Standard Random Forest Model on SMOTE-Balanced Historical Data
srf_2019_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

srf_2019_model.fit(
    historical_features_smote,
    historical_derates_smote
)

print("Standard Random Forest deployment model training complete.")

Standard Random Forest deployment model training complete.


In [192]:
# Generate Deployment Prediction Probabilities on Full 2019 Operational Data
srf_2019_probabilities = srf_2019_model.predict_proba(
    deployment_2019_features
)[:, 1]

In [193]:
# Apply Final Probability Threshold to Generate Deployment Predictions
srf_2019_predictions = (
    srf_2019_probabilities >= 0.20
).astype(int)

In [196]:
# Evaluate Standard Random Forest Deployment Performance on 2019 Operational Data
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score
)

srf_2019_cm = confusion_matrix(
    deployment_2019_derates,
    srf_2019_predictions
)

print(srf_2019_cm)

print(
    classification_report(
        deployment_2019_derates,
        srf_2019_predictions
    )
)

[[92876  2809]
 [   65    69]]
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     95685
           1       0.02      0.51      0.05       134

    accuracy                           0.97     95819
   macro avg       0.51      0.74      0.52     95819
weighted avg       1.00      0.97      0.98     95819



In [197]:
# Calculate Final Deployment Performance Metrics
srf_2019_f1 = f1_score(
    deployment_2019_derates,
    srf_2019_predictions
)

srf_2019_precision = precision_score(
    deployment_2019_derates,
    srf_2019_predictions
)

srf_2019_recall = recall_score(
    deployment_2019_derates,
    srf_2019_predictions
)

srf_2019_f1, srf_2019_precision, srf_2019_recall

(0.045816733067729085, 0.023974982626824185, 0.5149253731343284)

The deployment model successfully learned meaningful derate-related operational patterns from historical data and generalized reasonably well to unseen 2019 operational behavior, as demonstrated by the relatively strong recall performance. However, substantial overlap between derate and non-derate operational conditions produced very high false positive rates, limiting overall precision and reducing estimated operational savings under the assumed maintenance cost structure.

In [199]:
# Extract Operational Outcomes from Deployment Confusion Matrix
tn, fp, fn, tp = srf_2019_cm.ravel()

print("True Positives (Correct Derates):", tp)
print("False Positives (False Derates):", fp)
print("False Negatives (Missed Derates):", fn)
print("True Negatives (Correct Non-Derates):", tn)

True Positives (Correct Derates): 69
False Positives (False Derates): 2809
False Negatives (Missed Derates): 65
True Negatives (Correct Non-Derates): 92876


In [200]:
# Calculate Estimated Net Deployment Savings
srf_2019_net_savings = (
    tp * 4000
) - (
    fp * 500
)

srf_2019_net_savings

np.int64(-1128500)

The deployment model demonstrated meaningful sensitivity to future full derate events within unseen operational data, successfully identifying approximately half of all future derates. However, substantial overlap between derate and non-derate operational conditions produced a large number of false positive maintenance alerts, ultimately generating negative estimated net operational savings under the assumed deployment cost structure.

## Balanced Random Forest Deployment Evaluation

Balanced Random Forest was evaluated as a second deployment model because it handles severe class imbalance internally during tree construction. Unlike the Standard Random Forest model, this approach does not require SMOTE and may provide a different balance between catching future derates and controlling false positive maintenance alerts.

In [202]:
# Train Balanced Random Forest Model on Historical Pre-2019 Data
from imblearn.ensemble import BalancedRandomForestClassifier

brf_2019_model = BalancedRandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    random_state=321,
    n_jobs=-1
)

brf_2019_model.fit(
    historical_features,
    historical_derates
)

print("Balanced Random Forest deployment model training complete.")

Balanced Random Forest deployment model training complete.


In [204]:
# Generate Balanced Random Forest Deployment Prediction Probabilities
brf_2019_probabilities = brf_2019_model.predict_proba(
    deployment_2019_features
)[:, 1]

In [207]:
# Apply Final Deployment Probability Threshold
brf_2019_predictions = (
    brf_2019_probabilities >= 0.7
).astype(int)

In [208]:
# Evaluate Balanced Random Forest Deployment Performance
brf_2019_cm = confusion_matrix(
    deployment_2019_derates,
    brf_2019_predictions
)

print(brf_2019_cm)

print(
    classification_report(
        deployment_2019_derates,
        brf_2019_predictions
    )
)

[[89998  5687]
 [   39    95]]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97     95685
           1       0.02      0.71      0.03       134

    accuracy                           0.94     95819
   macro avg       0.51      0.82      0.50     95819
weighted avg       1.00      0.94      0.97     95819



In [209]:
# Calculate Balanced Random Forest Deployment Metrics
brf_2019_f1 = f1_score(
    deployment_2019_derates,
    brf_2019_predictions
)

brf_2019_precision = precision_score(
    deployment_2019_derates,
    brf_2019_predictions
)

brf_2019_recall = recall_score(
    deployment_2019_derates,
    brf_2019_predictions
)

brf_2019_f1, brf_2019_precision, brf_2019_recall

(0.03211629479377958, 0.016430300933932895, 0.7089552238805971)

Substantial operational overlap exists between derate and non-derate conditions, limiting model separability despite meaningful predictive signal detection capability.

In [211]:
# Extract Balanced Random Forest Operational Outcomes
tn_brf, fp_brf, fn_brf, tp_brf = (
    brf_2019_cm.ravel()
)

print("True Positives:", tp_brf)
print("False Positives:", fp_brf)
print("False Negatives:", fn_brf)
print("True Negatives:", tn_brf)

True Positives: 95
False Positives: 5687
False Negatives: 39
True Negatives: 89998


In [212]:
# Calculate Balanced Random Forest Net Deployment Savings
brf_2019_net_savings = (
    tp_brf * 4000
) - (
    fp_brf * 500
)

brf_2019_net_savings

np.int64(-2463500)

In [213]:
# Compare Final Deployment Performance Between SRF and BRF
deployment_model_comparison = pd.DataFrame({

    'Model': [
        'Standard Random Forest',
        'Balanced Random Forest'
    ],

    'F1_Score': [
        srf_2019_f1,
        brf_2019_f1
    ],

    'Precision': [
        srf_2019_precision,
        brf_2019_precision
    ],

    'Recall': [
        srf_2019_recall,
        brf_2019_recall
    ],

    'Correct_Derates_Caught_TP': [
        tp,
        tp_brf
    ],

    'False_Positive_Alerts_FP': [
        fp,
        fp_brf
    ],

    'Missed_Derates_FN': [
        fn,
        fn_brf
    ],

    'Estimated_Net_Savings': [
        srf_2019_net_savings,
        brf_2019_net_savings
    ]
})

deployment_model_comparison

,Model,F1_Score,Precision,Recall,Correct_Derates_Caught_TP,False_Positive_Alerts_FP,Missed_Derates_FN,Estimated_Net_Savings
0,Standard Random Forest,0.045817,0.023975,0.514925,69,2809,65,-1128500
1,Balanced Random Forest,0.032116,0.016430,0.708955,95,5687,39,-2463500


# Final 2019 Deployment Evaluation, Cost Analysis, and Operational Conclusions:
The final deployment evaluation demonstrated that both the Standard Random Forest and Balanced Random Forest models successfully learned meaningful operational fault behavior patterns from historical pre-2019 data and generalized to the fully unseen 2019 operational dataset. Both models identified a substantial proportion of future full derate events, indicating that predictive signal exists within the operational diagnostic and fault-code features.

The Balanced Random Forest model achieved the highest derate sensitivity, successfully identifying approximately 71% of future derates, while the Standard Random Forest model provided a more balanced tradeoff between recall and precision. Specifically, the Standard Random Forest model achieved stronger F1 performance, higher precision, fewer false positive maintenance alerts, and lower estimated operational losses relative to the Balanced Random Forest approach.

However, despite meaningful derate detection capability, both deployment models generated large numbers of false positive maintenance alerts, resulting in negative estimated net operational savings under the assumed business cost structure. These findings suggest that substantial operational overlap exists between derate and non-derate conditions, limiting model separability despite the presence of meaningful predictive signal within the data.

Overall, this project demonstrates that predictive maintenance modeling for future full derate events is feasible using operational diagnostic and fault-code data, but achieving economically viable real-world deployment performance will likely require additional temporal feature engineering, sequential modeling approaches, richer operational context, and improved cost-sensitive optimization strategies to better balance derate detection and false positive reduction.
